# Tahap 3 — Training & Evaluasi: ResNet50

**Ruang lingkup notebook ini: ResNet50 saja.** Notebook ini disalin dari `train_efficientnet.ipynb`; yang
berubah hanya `MODEL_NAME = "resnet50"` dan beberapa teks. Seluruh logika training ada di modul
**`training.py`** yang arsitektur-agnostik (ResNet50 sudah terdaftar di `ARCHITECTURES`), jadi split,
preprocessing, hyperparameter, dan cara menghitung metrik **identik** dengan EfficientNet-B0. Itu syarat
supaya perbedaan hasil benar-benar berasal dari arsitektur, bukan dari kode.

Hasil ditulis ke `results/resnet50/` dalam format yang sama dengan `results/efficientnet_b0/`.

**Prasyarat:** folder `data_split/` hasil `prepare_data.ipynb` (dedup + split 70/15/15, seed 42) sudah ada
di folder repo dan jumlahnya 4221 / 905 / 905.

## 1. Setup

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

import data_pipeline as dp
import training as tr

MODEL_NAME = "resnet50"

# Palet: biru = train, oranye = validation. Dipakai konsisten di semua grafik.
C_TRAIN, C_VAL, C_ACCENT = "#2a78d6", "#eb6834", "#1baf7a"
C_GRID, C_TEXT = "#d8d7d2", "#52514e"
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True, "grid.color": C_GRID, "grid.linewidth": 0.6,
    "axes.edgecolor": C_GRID, "axes.labelcolor": C_TEXT,
    "xtick.color": C_TEXT, "ytick.color": C_TEXT,
    "axes.spines.top": False, "axes.spines.right": False,
})

dp.seed_everything(dp.SEED)
DEVICE = dp.get_device()
print(f"Device: {DEVICE}")
print(f"Torch : {torch.__version__}")

## 2. Tentang ResNet50

Bagian ini untuk bagian "Penjelasan arsitektur" di laporan. **Isi dengan kata-katamu sendiri** setelah
membaca hasilnya; poin di bawah hanya kerangka.

- **Masalah yang dipecahkan:** apa yang terjadi kalau jaringan dibuat makin dalam? Mengapa itu sulit?
- **Ide utama: skip connection (residual).** Apa yang dijumlahkan, dan mengapa itu membantu jaringan dalam
  tetap bisa dilatih?
- **Blok penyusun: bottleneck.** Susunannya 1×1 → 3×3 → 1×1; jelaskan fungsi tiap konvolusi. ResNet50
  tersusun dari blok berjumlah `[3, 4, 6, 3]` (terlihat di kode torchvision).
- **Ukuran.** 23.571.551 parameter dengan kepala 31 kelas (hasil `count_parameters`), bobot awal
  `IMAGENET1K_V2`.
- **Catatan jujur untuk analisis:** bobot V2 dilatih dengan resep yang lebih modern daripada V1 yang dipakai
  VGG16 dan EfficientNet-B0. Jika ResNet50 unggul, sebagian bisa berasal dari bobot awalnya, bukan hanya
  arsitekturnya.
- **Relevansi untuk dataset ini:** 31 kelas, ~136 citra train per kelas. Apa implikasinya untuk model
  berukuran sedang?

In [ ]:
# Konteks ukuran arsitektur (bobot tidak diunduh, hanya struktur)
baris = []
for name in tr.ARCHITECTURES:
    m = tr.build_model(name, num_classes=31, pretrained=False)
    info = tr.count_parameters(m)
    baris.append({"arsitektur": name,
                  "params (M)": round(info["total_params"] / 1e6, 1),
                  "ukuran (MB)": round(info["size_mb"], 1)})
    del m

konteks = pd.DataFrame(baris).sort_values("params (M)")
print(konteks.to_string(index=False))
print(f"\n(Baris selain {MODEL_NAME} hanya konteks ukuran; pelatihannya bagian anggota kelompok lain.)")

## 3. Data & Model

In [ ]:
CONFIG = tr.TrainConfig(
    epochs=5,
    lr=1e-4,
    weight_decay=1e-4,
    batch_size=32,
    patience=5,
    augment="basic",
    use_class_weights=True,
)

loaders, ds = dp.build_dataloaders(
    # num_workers rendah: mesin ini 8 GB, worker tambahan memicu swap yang
    # biayanya jauh lebih besar daripada keuntungan paralelisasi loading.
    batch_size=CONFIG.batch_size, num_workers=2, augment=CONFIG.augment
)
classes = ds["train"].classes

print(f"train/val/test : {len(ds['train'])}/{len(ds['val'])}/{len(ds['test'])}")
print(f"kelas          : {len(classes)}")
print(f"batch per epoch: {len(loaders['train'])}")
print(f"\nHyperparameter (harus sama untuk semua arsitektur yang dibandingkan):")
for k, v in vars(CONFIG).items():
    print(f"  {k:20s}: {v}")

In [ ]:
model = tr.build_model(MODEL_NAME, num_classes=len(classes), pretrained=True)
info = tr.count_parameters(model)

print(f"{MODEL_NAME}: {info['total_params']:,} parameter ({info['total_params']/1e6:.2f} M), "
      f"{info['size_mb']:.1f} MB")
HEAD_ATTR = tr.ARCHITECTURES[MODEL_NAME][2]   # 'fc' untuk ResNet, 'classifier' untuk EfficientNet/VGG
print(f"\nLayer klasifikasi setelah diganti ({HEAD_ATTR}):\n{getattr(model, HEAD_ATTR)}")

# Seluruh jaringan di-fine-tune, bukan hanya head. Dengan ~4200 citra train,
# fine-tuning penuh memberi akurasi lebih tinggi daripada membekukan backbone,
# dan lr kecil (1e-4) menjaga fitur pretrained tidak rusak di epoch awal.
print(f"\nParameter yang dilatih: {info['trainable_params']:,} (seluruh jaringan, fine-tuning penuh)")

## 4. Training

Konfigurasi: AdamW + cosine annealing, `CrossEntropyLoss` berbobot kelas (menangani imbalance 1.73x dari
Tahap 1), early stopping pada **val macro-F1** dengan patience 5. Konfigurasi ini **sama persis** dengan
EfficientNet-B0 (lihat sel Data & Model di atas).

Dipilih macro-F1, bukan akurasi, sebagai kriteria model terbaik: dengan kelas yang tidak seimbang,
akurasi bisa naik hanya karena model membaik di kelas besar sementara kelas kecil seperti `asam jawa`
(85 citra train) justru memburuk. Macro-F1 memberi bobot sama ke tiap kelas.

**Training dijalankan dari sel di bawah** (progress bar per epoch tampil di output sel). Karena bisa
berlangsung belasan sampai puluhan menit:

- jalankan `caffeinate -i` di tab Terminal lain supaya Mac tidak tidur, hentikan dengan `Ctrl+C` setelah selesai;
- tutup aplikasi berat (RAM 8 GB mudah penuh);
- jangan menutup notebook atau menghentikan kernel sebelum sel selesai.

Setelah selesai, hasil tersimpan di `results/resnet50/` dan `models/resnet50_best.pt`. Menjalankan notebook
ini lagi akan memuat hasil tersimpan itu tanpa melatih ulang (kecuali `RETRAIN = True`).

In [ ]:
RETRAIN = False   # True = latih ulang dari notebook (progress bar per epoch tampil di output sel)

sudah_ada = (Path("results") / MODEL_NAME / "history.json").exists()

if RETRAIN or not sudah_ada:
    if not sudah_ada:
        print("Hasil training belum ada. Melatih dari notebook...\n")
    t0 = time.perf_counter()
    hasil = tr.fit(model, loaders, ds, model_name=MODEL_NAME, config=CONFIG, device=DEVICE)
    print(f"\nTotal waktu training: {(time.perf_counter() - t0)/60:.1f} menit")
else:
    hasil = tr.load_run(MODEL_NAME, model, device=DEVICE)
    print(f"Memuat hasil training tersimpan (dari run sebelumnya).")
    print(f"Jalankan ulang dengan RETRAIN = True kalau ingin melatih dari awal.\n")

print(f"Epoch dijalankan : {len(hasil['history'])}")
print(f"Epoch terbaik    : {hasil['best_epoch']} (val macro-F1 {hasil['best_val_macro_f1']:.4f})")
print(f"Waktu training   : {hasil['total_train_time_s']/60:.1f} menit")
print(f"Checkpoint       : {hasil['checkpoint']}")

### 4.1 Kurva Belajar

In [ ]:
hist = pd.DataFrame(hasil["history"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].plot(hist["epoch"], hist["train_loss"], color=C_TRAIN, lw=2, label="train")
axes[0].plot(hist["epoch"], hist["val_loss"], color=C_VAL, lw=2, label="validation")
axes[0].set(xlabel="epoch", ylabel="loss", title="Loss")
axes[0].legend(frameon=False)

axes[1].plot(hist["epoch"], hist["train_acc"], color=C_TRAIN, lw=2, label="train")
axes[1].plot(hist["epoch"], hist["val_acc"], color=C_VAL, lw=2, label="validation")
axes[1].set(xlabel="epoch", ylabel="akurasi", title="Akurasi")
axes[1].legend(frameon=False)

axes[2].plot(hist["epoch"], hist["val_macro_f1"], color=C_VAL, lw=2)
best = hasil["best_epoch"]
best_f1 = hasil["best_val_macro_f1"]
axes[2].scatter([best], [best_f1], color=C_VAL, s=60, zorder=5)
axes[2].annotate(f"terbaik: epoch {best}\nF1 {best_f1:.4f}",
                 xy=(best, best_f1), xytext=(6, -28), textcoords="offset points",
                 fontsize=9, color=C_TEXT)
axes[2].set(xlabel="epoch", ylabel="macro-F1", title="Validation macro-F1 (kriteria early stopping)")

plt.tight_layout()
plt.show()

print(hist[["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "val_macro_f1", "train_time_s"]]
      .round(4).to_string(index=False))

gap = hist.loc[hist["epoch"] == best, "train_acc"].iloc[0] - hist.loc[hist["epoch"] == best, "val_acc"].iloc[0]
print(f"\nGap train-val akurasi di epoch terbaik: {gap:+.4f}")
print("Gap besar (>0.10) menandakan overfitting -> pertimbangkan augment='strong'.")

## 5. Evaluasi pada Test Set

Test set baru disentuh **sekali, di sini**, memakai checkpoint terbaik yang dipilih berdasarkan
validation. Memilih model berdasarkan skor test akan membuat angka test tidak lagi menjadi estimasi
jujur terhadap data baru.

In [ ]:
evalres = tr.evaluate(model, loaders, ds, model_name=MODEL_NAME, hasil_fit=hasil, device=DEVICE)
m = evalres["metrics"]

print(f"=== {MODEL_NAME} — hasil test ===\n")
ringkas = pd.DataFrame([
    ("Accuracy", m["accuracy"]),
    ("Top-5 accuracy", m["top5_accuracy"]),
    ("Precision (macro)", m["precision_macro"]),
    ("Recall (macro)", m["recall_macro"]),
    ("F1 (macro)", m["f1_macro"]),
    ("Precision (weighted)", m["precision_weighted"]),
    ("Recall (weighted)", m["recall_weighted"]),
    ("F1 (weighted)", m["f1_weighted"]),
], columns=["metrik", "nilai"])
print(ringkas.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print(f"\nSelisih val macro-F1 ({hasil['best_val_macro_f1']:.4f}) vs test macro-F1 ({m['f1_macro']:.4f}): "
      f"{m['f1_macro'] - hasil['best_val_macro_f1']:+.4f}")
print(f"Hasil disimpan ke: {evalres['output_dir']}")

### 5.1 Metrik per Kelas

In [ ]:
per_class = pd.DataFrame(evalres["per_class"]).sort_values("f1")

fig, ax = plt.subplots(figsize=(9, 9))
y = np.arange(len(per_class))
ax.barh(y, per_class["f1"], color=C_TRAIN, height=0.7)
ax.set_yticks(y)
ax.set_yticklabels(per_class["class"], fontsize=8)
ax.axvline(m["f1_macro"], color=C_VAL, lw=2, ls="--")
ax.annotate(f"macro-F1 {m['f1_macro']:.3f}", xy=(m["f1_macro"], len(per_class) - 1.5),
            xytext=(6, 0), textcoords="offset points", fontsize=9, color=C_VAL)
ax.set(xlabel="F1-score", xlim=(0, 1.02), title=f"F1 per kelas — {MODEL_NAME} (test set)")
ax.grid(axis="y", visible=False)
# Label langsung pada 5 kelas terlemah: bagian yang paling perlu dibaca angkanya
for i in range(5):
    ax.text(per_class["f1"].iloc[i] + 0.012, i, f"{per_class['f1'].iloc[i]:.2f}",
            va="center", fontsize=8, color=C_TEXT)
plt.tight_layout()
plt.show()

print("5 kelas terlemah:")
print(per_class.head(5)[["class", "precision", "recall", "f1", "support"]]
      .round(4).to_string(index=False))
print("\n5 kelas terkuat:")
print(per_class.tail(5)[["class", "precision", "recall", "f1", "support"]]
      .round(4).to_string(index=False))

In [ ]:
# Apakah kelas kecil memang lebih sulit? Relevan karena Tahap 1 menemukan
# imbalance 1.73x setelah dedup.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(per_class["support"], per_class["f1"], color=C_TRAIN, s=45, zorder=3)
for _, r in per_class.head(4).iterrows():
    ax.annotate(r["class"], xy=(r["support"], r["f1"]), xytext=(5, -3),
                textcoords="offset points", fontsize=8, color=C_TEXT)
korelasi = per_class["support"].corr(per_class["f1"])
ax.set(xlabel="jumlah citra test (support)", ylabel="F1-score",
       title=f"F1 vs ukuran kelas (korelasi Pearson r = {korelasi:.2f})")
plt.tight_layout()
plt.show()
print(f"Korelasi support vs F1: r = {korelasi:.3f}")
print("r mendekati 0 berarti kesulitan kelas lebih ditentukan kemiripan visual daripada jumlah data.")

### 5.2 Confusion Matrix

In [ ]:
cm = evalres["confusion_matrix"]
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9.5))
# Sequential satu hue (terang -> gelap) untuk besaran; bukan rainbow.
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_yticklabels(classes, fontsize=7)
ax.set(xlabel="prediksi", ylabel="label sebenarnya",
       title=f"Confusion matrix ternormalisasi per baris — {MODEL_NAME} (test)")
ax.grid(visible=False)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("proporsi dari kelas sebenarnya", color=C_TEXT)
plt.tight_layout()
plt.show()

# Daftar kebingungan terbesar lebih terbaca daripada mencari sel gelap satu per satu
salah = [(classes[i], classes[j], cm[i, j], cm_norm[i, j])
         for i in range(len(classes)) for j in range(len(classes))
         if i != j and cm[i, j] > 0]
salah.sort(key=lambda t: -t[2])

konfusi = pd.DataFrame(salah[:15], columns=["sebenarnya", "diprediksi", "n", "proporsi"])
print("15 kebingungan terbesar:")
print(konfusi.round(3).to_string(index=False))
print(f"\nTotal salah klasifikasi: {int(cm.sum() - np.trace(cm))} dari {int(cm.sum())} citra test")

### 5.3 Analisis Kesalahan

In [ ]:
probs, preds, trues = evalres["probs"], evalres["preds"], evalres["trues"]
paths = [p for p, _ in ds["test"].samples]

salah_idx = np.where(preds != trues)[0]
conf = probs[np.arange(len(preds)), preds]

print(f"Jumlah salah: {len(salah_idx)} ({len(salah_idx)/len(preds):.2%})")
print(f"Rata-rata confidence saat BENAR : {conf[preds == trues].mean():.4f}")
print(f"Rata-rata confidence saat SALAH : {conf[salah_idx].mean():.4f}")
print("\nModel yang terkalibrasi baik seharusnya kurang yakin ketika salah.")

# Kesalahan paling percaya diri = kandidat label salah atau kelas yang benar-benar mirip
urut = salah_idx[np.argsort(-conf[salah_idx])][:10]
fig, axes = plt.subplots(2, 5, figsize=(15, 6.6))
for ax, i in zip(axes.ravel(), urut):
    ax.imshow(Image.open(paths[i]).convert("RGB"))
    ax.set_title(f"asli: {classes[trues[i]]}\nprediksi: {classes[preds[i]]} ({conf[i]:.2f})",
                 fontsize=8, color=C_TEXT)
    ax.axis("off")
fig.suptitle("10 kesalahan dengan confidence tertinggi")
plt.tight_layout()
plt.show()

In [ ]:
# Prediksi benar dari model terbaik, sesuai permintaan tugas langkah 5
rng = np.random.default_rng(dp.SEED)
benar_idx = np.where(preds == trues)[0]
pilih = rng.choice(benar_idx, 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6.6))
for ax, i in zip(axes.ravel(), pilih):
    ax.imshow(Image.open(paths[i]).convert("RGB"))
    ax.set_title(f"{classes[preds[i]]}\nconfidence {conf[i]:.2f}", fontsize=8, color=C_TEXT)
    ax.axis("off")
fig.suptitle("Contoh prediksi benar")
plt.tight_layout()
plt.show()

## 6. Efisiensi Komputasi

In [ ]:
eff = pd.DataFrame([
    ("Jumlah parameter", f"{m['total_params']:,} ({m['total_params']/1e6:.2f} M)"),
    ("Ukuran model", f"{m['size_mb']:.1f} MB"),
    ("Epoch dijalankan", f"{m['epochs_run']} (terbaik: epoch {m['best_epoch']})"),
    ("Waktu training total", f"{m['total_train_time_s']/60:.1f} menit"),
    ("Rata-rata waktu/epoch", f"{m['mean_epoch_time_s']:.1f} detik"),
    ("Latensi inferensi", f"{m['infer_latency_ms_per_batch']:.1f} ms/batch "
                          f"({m['infer_latency_ms_per_image']:.2f} ms/citra)"),
    ("Throughput inferensi", f"{m['infer_throughput_img_per_s']:.0f} citra/detik"),
    ("Device", f"{m['device']} — {m['platform']}"),
], columns=["metrik", "nilai"])
print(eff.to_string(index=False))

print(f"\nAkurasi per juta parameter: {m['accuracy'] / (m['total_params']/1e6):.4f}")
print("Angka ini berguna untuk membandingkan efisiensi arsitektur di tahap perbandingan kelompok.")

## 7. Ekspor Hasil untuk Perbandingan Kelompok

In [ ]:
out_dir = Path(evalres["output_dir"])
print(f"Isi {out_dir}:")
for f in sorted(out_dir.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size/1024:8.1f} KB")

print(f"""
Anggota kelompok lain menjalankan hal yang sama dengan mengganti satu baris:

    MODEL_NAME = "resnet18"      # atau alexnet / vgg16 / mobilenet_v2 / resnet50

SYARAT agar perbandingannya sahih — semuanya sudah otomatis lewat modul:
  - data_split/ yang sama (hasil dedup, {m['n_train']}/{m['n_val']}/{m['n_test']})
  - dp.build_dataloaders() yang sama (224x224, normalisasi ImageNet, augment="{CONFIG.augment}")
  - TrainConfig yang sama (lr={CONFIG.lr}, epochs={CONFIG.epochs}, batch={CONFIG.batch_size})
  - metrik dihitung oleh tr.evaluate() yang sama

CATATAN: VGG16 tidak muat di batch 32 pada mesin 8 GB (out of memory). Kalau anggota lain
harus menurunkan batch size, hal itu WAJIB dicatat di laporan — waktu training antar arsitektur
jadi tidak apple-to-apple, meski metrik akurasinya tetap bisa dibandingkan.

Tahap perbandingan nanti cukup memanggil:
    tr.load_results("efficientnet_b0"), tr.load_results("resnet18"), ...
""")

In [ ]:
# Ringkasan satu baris, siap digabung jadi tabel perbandingan di laporan
ringkasan_baris = {
    "arsitektur": MODEL_NAME,
    "params_M": round(m["total_params"] / 1e6, 2),
    "accuracy": round(m["accuracy"], 4),
    "precision_macro": round(m["precision_macro"], 4),
    "recall_macro": round(m["recall_macro"], 4),
    "f1_macro": round(m["f1_macro"], 4),
    "train_time_min": round(m["total_train_time_s"] / 60, 1),
    "infer_ms_per_image": round(m["infer_latency_ms_per_image"], 2),
}
print(pd.DataFrame([ringkasan_baris]).to_string(index=False))

Path("results").mkdir(exist_ok=True)
pd.DataFrame([ringkasan_baris]).to_csv(f"results/{MODEL_NAME}/summary_row.csv", index=False)
print(f"\nDisimpan: results/{MODEL_NAME}/summary_row.csv")

## 8. Ringkasan

Isi setelah melihat hasil di atas — poin-poin ini yang masuk ke laporan:

- **Performa.** Akurasi dan macro-F1 test, serta selisihnya terhadap validation. [ISI setelah hasil keluar]
- **Kelas tersulit.** Lihat 5 kelas ber-F1 terendah dan tabel kebingungan terbesar. Periksa apakah pasangan
  yang tertukar memang mirip secara visual. [ISI]
- **Keterbatasan data yang sudah diketahui dari Tahap 1:** 11% citra ter-upsample dari resolusi di bawah
  224 px, dan kelas `asam jawa` hanya punya 85 citra train setelah dedup. Keduanya membatasi akurasi
  maksimum dan perlu disebut di laporan.
- **Efisiensi.** Jumlah parameter, waktu training, dan latensi inferensi dari tabel Bagian 6, dibandingkan
  dengan EfficientNet-B0 dan VGG16. Sebutkan perangkat yang dipakai (`device` di `metrics.json`); waktu
  antar mesin berbeda tidak apple-to-apple. [ISI]
- **Perbandingan.** Apakah ResNet50 lebih baik daripada EfficientNet-B0, dan dengan biaya apa? Ingat catatan
  bobot V2 vs V1. [ISI setelah hasil ketiga model masuk]